# Retailrocket recsys

This notebook is designed to train a recsys model on Retailrocket dataset.

In [ ]:
import sys
import csv
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from data import download_kaggle_dataset, preprocess_retailrocket_data

Download the Retailrocket dataset into the local `data/` folder so the preprocessing step can work with the raw CSV files.

In [ ]:
dataset_name = "retailrocket/ecommerce-dataset"
output_path = download_kaggle_dataset(dataset_name, output_dir=project_root / "data")
print(f"Downloaded dataset into: {output_path}")
sorted(str(path.relative_to(output_path)) for path in output_path.rglob("*") if path.is_file())[:20]

Run the shared preprocessing helper once to prepare the derived Retailrocket tables.

`preprocess_retailrocket_data(...)` currently does four things:
- builds `category_lookup.csv` from `category_tree.csv`
- computes the category leaf-depth summary returned as `leaf_depth_statistics`
- merges `item_properties_part1.csv` and `item_properties_part2.csv` into `item_properties.csv`
- splits the merged item properties into 3 different item - property files.
- generate user_events and user_item files.
- The following should take 10 minutes and only needs to run once.

In [ ]:
preprocess_outputs = preprocess_retailrocket_data(output_path)
category_lookup_path = preprocess_outputs["category_lookup_path"]
leaf_depth_stats = preprocess_outputs["leaf_depth_statistics"]
merged_item_properties_path = preprocess_outputs["merged_item_properties_path"]

print(f"Generated category lookup table at: {category_lookup_path}")
print(f"Merged item properties file at: {merged_item_properties_path}")
print("Leaf depth statistics:")
for depth, count in leaf_depth_stats.items():
    print(f"depth={depth}, count={count}")

with category_lookup_path.open(newline="", encoding="utf-8") as handle:
    preview_rows = list(csv.reader(handle))[:10]

preview_rows

Build or load the saved all-item embedding table produced by `model.get_all_items_embedding(...)`.

Load the iterable user-item dataset from `user_item.csv` and split it into `train_set` and `test_set` with the dataset's built-in `split(...)` helper.

The following data loading would take 3 minutes to load all the data.

In [ ]:
from pathlib import Path

from data import UserItemEmbeddingIterableDataset
from model import initialize_item_embedding_resources

user_item_path = Path(output_path) / "user_item.csv"

state = initialize_item_embedding_resources(dataset_path=output_path)

positive_negative_ratio = 1.0

dataset = UserItemEmbeddingIterableDataset(
    user_item_path=user_item_path,
    resources=state.resources,
    token_transformer=state.token_transformer,
    item_projection=state.item_projection,
    user_projection=state.user_projection,
    available_filter=1,
    positive_negative_ratio=positive_negative_ratio,
    device=state.device,
)

train_set, test_set = dataset.split(train_size=0.8, seed=42)

print(f"train rows: {len(train_set)}")
print(f"test rows: {len(test_set)}")


In [ ]:
from torch.optim import AdamW
from model import FactorizationMachines, load_model, save_model

learning_rate = 1e-5

fm_model = FactorizationMachines(
    embedding_dim=state.item_projection.out_features,
    latent_dim=16,
).to(state.device)
optimizer = AdamW(fm_model.parameters(), lr=learning_rate)
checkpoint_dir = Path(output_path) / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
%env WANDB_API_KEY=xxx

In [ ]:
import torch
from tqdm.auto import tqdm
import wandb
from torch.utils.data import DataLoader
from data import collate_user_item_embedding_batch

def get_total_model_parameter_l2_norm(*modules):
    total_squared_norm = 0.0
    for module in modules:
        for parameter in module.parameters():
            parameter_norm = parameter.detach().float().norm(2).item()
            total_squared_norm += parameter_norm * parameter_norm
    return total_squared_norm ** 0.5


batch_size = 256
epochs = 3
wandb_project = "recsys-retailrocket"

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    collate_fn=collate_user_item_embedding_batch,
)
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    collate_fn=collate_user_item_embedding_batch,
)

wandb_run = wandb.init(
    project=wandb_project,
    config={
        "batch_size": batch_size,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "fm_latent_dim": fm_model.latent_dim,
        "embedding_dim": fm_model.embedding_dim,
    },
)

try:
    global_step = 0
    for epoch in range(epochs):
        fm_model.train()
        train_loss_total = 0.0
        train_correct = 0
        train_example_count = 0

        train_progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [train]")
        for batch in train_progress:
            optimizer.zero_grad()
            logits, loss = fm_model.forward_batch(batch)
            loss.backward()
            optimizer.step()

            labels = batch[2]
            predictions = (torch.sigmoid(logits) >= 0.5).to(labels.dtype)
            batch_size_actual = labels.numel()
            train_loss_total += loss.item() * batch_size_actual
            train_correct += (predictions == labels).sum().item()
            train_example_count += batch_size_actual

            avg_train_loss_so_far = train_loss_total / max(train_example_count, 1)
            avg_train_accuracy_so_far = train_correct / max(train_example_count, 1)
            total_model_l2_norm = get_total_model_parameter_l2_norm(
                fm_model,
                state.token_transformer,
                state.item_projection,
                state.user_projection,
            )
            global_step += 1
            wandb.log(
                {
                    "train/step_loss": loss.item(),
                    "train/step_accuracy": (predictions == labels).float().mean().item(),
                    "train/running_loss": avg_train_loss_so_far,
                    "train/running_accuracy": avg_train_accuracy_so_far,
                    "model/total_l2_norm": total_model_l2_norm,
                    "train/epoch": epoch + 1,
                    "trainer/global_step": global_step,
                    "trainer/examples_seen": train_example_count,
                },
                step=global_step,
            )
            train_progress.set_postfix(
                loss=f"{avg_train_loss_so_far:.4f}",
                accuracy=f"{avg_train_accuracy_so_far:.4f}",
                l2_norm=f"{total_model_l2_norm:.4f}",
            )

        fm_model.eval()
        test_loss_total = 0.0
        test_correct = 0
        test_example_count = 0
        test_true_positive = 0
        test_false_negative = 0
        test_false_positive = 0

        with torch.no_grad():
            test_progress = tqdm(test_loader, desc=f"Epoch {epoch + 1}/{epochs} [test]")
            for batch in test_progress:
                logits, loss = fm_model.forward_batch(batch)
                labels = batch[2]
                predictions = (torch.sigmoid(logits) >= 0.5).to(labels.dtype)
                batch_size_actual = labels.numel()
                test_loss_total += loss.item() * batch_size_actual
                test_correct += (predictions == labels).sum().item()
                test_example_count += batch_size_actual
                test_true_positive += ((predictions == 1) & (labels == 1)).sum().item()
                test_false_negative += ((predictions == 0) & (labels == 1)).sum().item()
                test_false_positive += ((predictions == 1) & (labels == 0)).sum().item()

                avg_test_loss_so_far = test_loss_total / max(test_example_count, 1)
                avg_test_accuracy_so_far = test_correct / max(test_example_count, 1)
                test_progress.set_postfix(
                    loss=f"{avg_test_loss_so_far:.4f}",
                    accuracy=f"{avg_test_accuracy_so_far:.4f}",
                )

        avg_train_loss = train_loss_total / max(train_example_count, 1)
        avg_train_accuracy = train_correct / max(train_example_count, 1)
        avg_test_loss = test_loss_total / max(test_example_count, 1)
        avg_test_accuracy = test_correct / max(test_example_count, 1)
        test_recall = test_true_positive / max(test_true_positive + test_false_negative, 1)
        test_precision = test_true_positive / max(test_true_positive + test_false_positive, 1)
        test_f1 = 0.0 if (test_precision + test_recall) == 0 else 2 * test_precision * test_recall / (test_precision + test_recall)
        total_model_l2_norm = get_total_model_parameter_l2_norm(
            fm_model,
            state.token_transformer,
            state.item_projection,
            state.user_projection,
        )

        wandb.log(
            {
                "epoch": epoch + 1,
                "train/loss": avg_train_loss,
                "train/accuracy": avg_train_accuracy,
                "test/loss": avg_test_loss,
                "test/accuracy": avg_test_accuracy,
                "test/recall": test_recall,
                "test/f1": test_f1,
                "model/total_l2_norm": total_model_l2_norm,
                "trainer/global_step": global_step,
            }
            ,
            step=global_step,
        )
        print(
            f"epoch {epoch + 1}: train_loss={avg_train_loss:.4f}, train_accuracy={avg_train_accuracy:.4f}, "
            f"test_loss={avg_test_loss:.4f}, test_accuracy={avg_test_accuracy:.4f}, test_recall={test_recall:.4f}, "
            f"test_f1={test_f1:.4f}, model_total_l2_norm={total_model_l2_norm:.4f}"
        )
        epoch_checkpoint_path = checkpoint_dir / f"model_epoch_{epoch + 1}.pt"
        latest_checkpoint_path = checkpoint_dir / "model_latest.pt"
        save_model(epoch_checkpoint_path, state=state, fm_model=fm_model)
        save_model(latest_checkpoint_path, state=state, fm_model=fm_model)
        print(f"saved checkpoints: {epoch_checkpoint_path} and {latest_checkpoint_path}")
finally:
    wandb.finish()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

loaded_bundle = load_model(latest_checkpoint_path, device=state.device)
test_set.token_transformer = loaded_bundle.token_transformer
test_set.item_projection = loaded_bundle.item_projection
test_set.user_projection = loaded_bundle.user_projection

reloaded_test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    collate_fn=collate_user_item_embedding_batch,
)

loaded_bundle.fm_model.eval()
reloaded_test_loss_total = 0.0
reloaded_test_correct = 0
reloaded_test_example_count = 0
reloaded_test_true_positive = 0
reloaded_test_false_negative = 0
reloaded_test_false_positive = 0
reloaded_probabilities = []
reloaded_labels = []

with torch.no_grad():
    reloaded_test_progress = tqdm(reloaded_test_loader, desc="Reloaded checkpoint [test]")
    for batch in reloaded_test_progress:
        logits, loss = loaded_bundle.fm_model.forward_batch(batch)
        if not torch.isfinite(loss):
            raise ValueError(
                "Encountered non-finite evaluation loss. "
                "Check preprocessed numeric feature files for NaN/inf values."
            )
        labels = batch[2]
        predictions = (torch.sigmoid(logits) >= 0.5).to(labels.dtype)
        batch_size_actual = labels.numel()
        reloaded_test_loss_total += loss.item() * batch_size_actual
        reloaded_test_correct += (predictions == labels).sum().item()
        reloaded_test_example_count += batch_size_actual
        reloaded_test_true_positive += ((predictions == 1) & (labels == 1)).sum().item()
        reloaded_test_false_negative += ((predictions == 0) & (labels == 1)).sum().item()
        reloaded_test_false_positive += ((predictions == 1) & (labels == 0)).sum().item()
        reloaded_probabilities.append(torch.sigmoid(logits).detach().cpu())
        reloaded_labels.append(labels.detach().cpu().to(torch.int64))

        avg_reloaded_test_loss_so_far = reloaded_test_loss_total / max(reloaded_test_example_count, 1)
        avg_reloaded_test_accuracy_so_far = reloaded_test_correct / max(reloaded_test_example_count, 1)
        reloaded_test_progress.set_postfix(
            loss=f"{avg_reloaded_test_loss_so_far:.4f}",
            accuracy=f"{avg_reloaded_test_accuracy_so_far:.4f}",
        )

reloaded_test_loss = reloaded_test_loss_total / max(reloaded_test_example_count, 1)
reloaded_test_accuracy = reloaded_test_correct / max(reloaded_test_example_count, 1)
reloaded_test_recall = reloaded_test_true_positive / max(reloaded_test_true_positive + reloaded_test_false_negative, 1)
reloaded_test_precision = reloaded_test_true_positive / max(reloaded_test_true_positive + reloaded_test_false_positive, 1)
reloaded_test_f1 = 0.0 if (reloaded_test_precision + reloaded_test_recall) == 0 else 2 * reloaded_test_precision * reloaded_test_recall / (reloaded_test_precision + reloaded_test_recall)
reloaded_probabilities = torch.cat(reloaded_probabilities)
reloaded_labels = torch.cat(reloaded_labels).to(torch.bool)

threshold_rows = []
positive_count = reloaded_labels.sum().item()
negative_count = (~reloaded_labels).sum().item()
if positive_count == 0 or negative_count == 0:
    raise ValueError("ROC requires both positive and negative labels in the evaluation set.")

for threshold in torch.linspace(0.0, 1.0, steps=101):
    threshold_predictions = reloaded_probabilities >= threshold
    true_positive = (threshold_predictions & reloaded_labels).sum().item()
    false_positive = (threshold_predictions & ~reloaded_labels).sum().item()
    false_negative = ((~threshold_predictions) & reloaded_labels).sum().item()
    true_negative = ((~threshold_predictions) & ~reloaded_labels).sum().item()

    threshold_precision = true_positive / max(true_positive + false_positive, 1)
    threshold_recall = true_positive / max(positive_count, 1)
    threshold_accuracy = (true_positive + true_negative) / max(positive_count + negative_count, 1)
    threshold_f1 = 0.0 if (threshold_precision + threshold_recall) == 0 else 2 * threshold_precision * threshold_recall / (threshold_precision + threshold_recall)
    threshold_rows.append(
        {
            "threshold": float(threshold.item()),
            "fpr": false_positive / max(negative_count, 1),
            "tpr": threshold_recall,
            "accuracy": threshold_accuracy,
            "precision": threshold_precision,
            "recall": threshold_recall,
            "f1": threshold_f1,
        }
    )

roc_points = sorted(
    {(row["fpr"], row["tpr"]) for row in threshold_rows} | {(0.0, 0.0), (1.0, 1.0)},
    key=lambda point: (point[0], point[1]),
)
fpr_values = np.asarray([point[0] for point in roc_points], dtype=np.float64)
tpr_values = np.asarray([point[1] for point in roc_points], dtype=np.float64)
roc_auc = float(np.trapz(tpr_values, fpr_values))
best_threshold_row = max(threshold_rows, key=lambda row: (row["f1"], row["accuracy"], -abs(row["threshold"] - 0.5)))

print(
    "reloaded checkpoint test metrics: "
    f"loss={reloaded_test_loss:.4f}, accuracy={reloaded_test_accuracy:.4f}, "
    f"recall={reloaded_test_recall:.4f}, precision={reloaded_test_precision:.4f}, "
    f"f1={reloaded_test_f1:.4f}"
)
print(
    "best threshold by F1: "
    f"threshold={best_threshold_row['threshold']:.2f}, accuracy={best_threshold_row['accuracy']:.4f}, "
    f"precision={best_threshold_row['precision']:.4f}, recall={best_threshold_row['recall']:.4f}, "
    f"f1={best_threshold_row['f1']:.4f}"
)

plt.figure(figsize=(6, 6))
plt.plot(fpr_values, tpr_values, linewidth=2, label=f"ROC curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="gray", label="Random baseline")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.05)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Reloaded Checkpoint ROC Curve")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import csv

cold_start_positive_row_set = set(test_set.positive_rows)
cold_start_positive_rows = []
cold_start_negative_rows = []
all_test_rows = list(test_set.positive_rows) + list(test_set.negative_rows)
candidate_user_ids = sorted({visitor_id for visitor_id, _, _ in all_test_rows})
existing_positive_samples = set(test_set.positive_rows)
existing_negative_samples = set(test_set.negative_rows)

with test_set.user_item_path.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        if test_set.available_filter is not None and str(row.get("available", "")) != test_set.available_filter:
            continue
        if str(row.get("is_cold_start_item", "")) != "1":
            continue

        mapped_item_id = test_set.resources.item_id_by_original_id.get(int(row["itemid"]))
        if mapped_item_id is None:
            continue

        sample = (int(row["visitorid"]), mapped_item_id, int(row["label"]))
        if sample in cold_start_positive_row_set:
            cold_start_positive_rows.append(sample)

for sample_index, (visitor_id, item_id, _) in enumerate(cold_start_positive_rows):
    replacement_user_id = None
    for offset in range(len(candidate_user_ids)):
        candidate_user_id = candidate_user_ids[(sample_index + offset) % len(candidate_user_ids)]
        if candidate_user_id == visitor_id:
            continue
        candidate_negative_sample = (candidate_user_id, item_id, 0)
        candidate_positive_sample = (candidate_user_id, item_id, 1)
        if candidate_negative_sample in existing_negative_samples:
            continue
        if candidate_positive_sample in existing_positive_samples:
            continue
        replacement_user_id = candidate_user_id
        break

    if replacement_user_id is None:
        raise ValueError(f"Could not find a replacement user for cold-start item {item_id}.")

    cold_start_negative_rows.append((replacement_user_id, item_id, 0))

cold_start_test_set = UserItemEmbeddingIterableDataset._from_rows(
    user_item_path=test_set.user_item_path,
    resources=test_set.resources,
    token_transformer=loaded_bundle.token_transformer,
    item_projection=loaded_bundle.item_projection,
    user_projection=loaded_bundle.user_projection,
    device=test_set.device,
    positive_negative_ratio=None,
    samples_per_epoch=None,
    shuffle=False,
    available_filter=test_set.available_filter,
    is_cold_start_item_filter="1",
    positive_rows=tuple(cold_start_positive_rows),
    negative_rows=tuple(cold_start_negative_rows),
)

print(
    f"cold-start test rows: {len(cold_start_test_set)} "
    f"(positive={len(cold_start_positive_rows)}, negative={len(cold_start_negative_rows)})"
)

cold_start_test_loader = DataLoader(
    cold_start_test_set,
    batch_size=batch_size,
    collate_fn=collate_user_item_embedding_batch,
)

loaded_bundle.fm_model.eval()
cold_start_test_loss_total = 0.0
cold_start_test_correct = 0
cold_start_test_example_count = 0
cold_start_test_true_positive = 0
cold_start_test_false_negative = 0
cold_start_test_false_positive = 0
cold_start_probabilities = []
cold_start_labels = []

with torch.no_grad():
    cold_start_test_progress = tqdm(cold_start_test_loader, desc="Cold-start checkpoint [test]")
    for batch in cold_start_test_progress:
        logits, loss = loaded_bundle.fm_model.forward_batch(batch)
        if not torch.isfinite(loss):
            raise ValueError(
                "Encountered non-finite evaluation loss. "
                "Check preprocessed numeric feature files for NaN/inf values."
            )
        labels = batch[2]
        predictions = (torch.sigmoid(logits) >= 0.5).to(labels.dtype)
        batch_size_actual = labels.numel()
        cold_start_test_loss_total += loss.item() * batch_size_actual
        cold_start_test_correct += (predictions == labels).sum().item()
        cold_start_test_example_count += batch_size_actual
        cold_start_test_true_positive += ((predictions == 1) & (labels == 1)).sum().item()
        cold_start_test_false_negative += ((predictions == 0) & (labels == 1)).sum().item()
        cold_start_test_false_positive += ((predictions == 1) & (labels == 0)).sum().item()
        cold_start_probabilities.append(torch.sigmoid(logits).detach().cpu())
        cold_start_labels.append(labels.detach().cpu().to(torch.int64))

        avg_cold_start_test_loss_so_far = cold_start_test_loss_total / max(cold_start_test_example_count, 1)
        avg_cold_start_test_accuracy_so_far = cold_start_test_correct / max(cold_start_test_example_count, 1)
        cold_start_test_progress.set_postfix(
            loss=f"{avg_cold_start_test_loss_so_far:.4f}",
            accuracy=f"{avg_cold_start_test_accuracy_so_far:.4f}",
        )

cold_start_test_loss = cold_start_test_loss_total / max(cold_start_test_example_count, 1)
cold_start_test_accuracy = cold_start_test_correct / max(cold_start_test_example_count, 1)
cold_start_test_recall = cold_start_test_true_positive / max(cold_start_test_true_positive + cold_start_test_false_negative, 1)
cold_start_test_precision = cold_start_test_true_positive / max(cold_start_test_true_positive + cold_start_test_false_positive, 1)
cold_start_test_f1 = 0.0 if (cold_start_test_precision + cold_start_test_recall) == 0 else 2 * cold_start_test_precision * cold_start_test_recall / (cold_start_test_precision + cold_start_test_recall)
cold_start_probabilities = torch.cat(cold_start_probabilities)
cold_start_labels = torch.cat(cold_start_labels).to(torch.bool)

cold_start_threshold_rows = []
cold_start_positive_count = cold_start_labels.sum().item()
cold_start_negative_count = (~cold_start_labels).sum().item()
if cold_start_positive_count == 0 or cold_start_negative_count == 0:
    raise ValueError("ROC requires both positive and negative labels in the cold-start evaluation set.")

for threshold in torch.linspace(0.0, 1.0, steps=101):
    threshold_predictions = cold_start_probabilities >= threshold
    true_positive = (threshold_predictions & cold_start_labels).sum().item()
    false_positive = (threshold_predictions & ~cold_start_labels).sum().item()
    false_negative = ((~threshold_predictions) & cold_start_labels).sum().item()
    true_negative = ((~threshold_predictions) & ~cold_start_labels).sum().item()

    threshold_precision = true_positive / max(true_positive + false_positive, 1)
    threshold_recall = true_positive / max(cold_start_positive_count, 1)
    threshold_accuracy = (true_positive + true_negative) / max(cold_start_positive_count + cold_start_negative_count, 1)
    threshold_f1 = 0.0 if (threshold_precision + threshold_recall) == 0 else 2 * threshold_precision * threshold_recall / (threshold_precision + threshold_recall)
    cold_start_threshold_rows.append(
        {
            "threshold": float(threshold.item()),
            "fpr": false_positive / max(cold_start_negative_count, 1),
            "tpr": threshold_recall,
            "accuracy": threshold_accuracy,
            "precision": threshold_precision,
            "recall": threshold_recall,
            "f1": threshold_f1,
        }
    )

cold_start_roc_points = sorted(
    {(row["fpr"], row["tpr"]) for row in cold_start_threshold_rows} | {(0.0, 0.0), (1.0, 1.0)},
    key=lambda point: (point[0], point[1]),
)
cold_start_fpr_values = np.asarray([point[0] for point in cold_start_roc_points], dtype=np.float64)
cold_start_tpr_values = np.asarray([point[1] for point in cold_start_roc_points], dtype=np.float64)
cold_start_roc_auc = float(np.trapz(cold_start_tpr_values, cold_start_fpr_values))
cold_start_best_threshold_row = max(cold_start_threshold_rows, key=lambda row: (row["f1"], row["accuracy"], -abs(row["threshold"] - 0.5)))

print(
    "cold-start checkpoint test metrics: "
    f"loss={cold_start_test_loss:.4f}, accuracy={cold_start_test_accuracy:.4f}, "
    f"recall={cold_start_test_recall:.4f}, precision={cold_start_test_precision:.4f}, "
    f"f1={cold_start_test_f1:.4f}"
)
print(
    "cold-start best threshold by F1: "
    f"threshold={cold_start_best_threshold_row['threshold']:.2f}, accuracy={cold_start_best_threshold_row['accuracy']:.4f}, "
    f"precision={cold_start_best_threshold_row['precision']:.4f}, recall={cold_start_best_threshold_row['recall']:.4f}, "
    f"f1={cold_start_best_threshold_row['f1']:.4f}"
)

plt.figure(figsize=(6, 6))
plt.plot(cold_start_fpr_values, cold_start_tpr_values, linewidth=2, label=f"ROC curve (AUC = {cold_start_roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="gray", label="Random baseline")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.05)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cold-Start Reloaded Checkpoint ROC Curve")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()
